In [13]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import BertTokenizer, BertModel


In [14]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
MAX_EPOCHS = 4  # Maximum epochs for early stopping
PATIENCE = 3     # Patience for early stopping

tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

print(f"Using device: {DEVICE}")

Using device: cuda


In [15]:
ds = load_dataset("higopires/RePro-categories-multilabel")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,review_text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,INADEQUADA,ANUNCIO
0,"Aparelho muito bom, confiável e com valor aqui...",0,0,1,0,0,0
1,"A história é muito boa, porém o autor ""enrolou...",0,0,1,0,0,0
2,"Entrega rápida, produto muito bom Amei. Pratic...",1,0,1,0,0,0
3,Produto otimo so falta o carregador da maquina...,0,0,1,1,0,0
4,a proteção anti queda não é boa se cair de fr...,0,0,1,0,0,0
...,...,...,...,...,...,...,...
7997,amei o produto. chegou no prazo e em perfeito ...,1,0,1,1,0,0
7998,Ótima embalagem. Produto entregue no prazo. Re...,1,0,1,1,0,0
7999,"ótimo produto, super recomendo .,Entrega bem r...",1,0,1,0,0,0
8000,"Veio tudo certinho, dentro do prazo e o produt...",1,0,1,1,0,0


In [4]:
train_df = train_df[train_df['INADEQUADA'] == 0].reset_index(drop=True)
val_df = val_df[val_df['INADEQUADA'] == 0].reset_index(drop=True)
test_df = test_df[test_df['INADEQUADA'] == 0].reset_index(drop=True)

train_df = train_df.drop(columns=['INADEQUADA'])
val_df = val_df.drop(columns=['INADEQUADA'])
test_df = test_df.drop(columns=['INADEQUADA'])

train_df = train_df.rename(columns={'review_text': 'text'})
val_df = val_df.rename(columns={'review_text': 'text'})
test_df = test_df.rename(columns={'review_text': 'text'})

train_df

,text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,ANUNCIO
0,"QUEM comprou recebeu 3,2 l family ????????? qu...",0,1,0,0,0
1,"Sobre o Produto é Excelente Recomendo , Americ...",1,1,1,0,0
2,"Adorei o Kit, entregue dentro do prazo, não ve...",1,0,1,1,0
3,"esperava mais, mas me serviu. é muito leve e s...",0,0,1,0,0
4,"Muito bom, o único problema é que veio azul ou...",0,0,1,1,1
...,...,...,...,...,...,...
73,"Comprei o kit e chegou rápido , porém sem lacr...",1,1,0,1,0
74,"Produto muito bom, ótimo custo benefício, a ja...",0,0,1,0,0
75,"É um produto que supre a minha necessidade, só...",0,0,1,0,0
76,O produto é de ótima qualidade e bem reforçado...,0,1,1,0,0


In [5]:
class MultiLabelClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        """
        Args:
            texts: List or array of text samples
            labels: 2D array of shape (num_samples, num_classes) with binary indicators (0 or 1)
            tokenizer: Pretrained tokenizer (e.g., BertTokenizer)
            max_len: Maximum sequence length
        """
        self.texts = texts
        self.labels = labels  # Shape: (num_samples, num_classes)
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]  # Shape: (num_classes,)
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)  # Binary vector for multilabel
        }

In [6]:
class BertForMultiLabelClassification(nn.Module):
    def __init__(self, num_classes):
        super(BertForMultiLabelClassification, self).__init__()
        self.bert = BertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output logits for each class
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits for BCEWithLogitsLoss

In [7]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [8]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, save_path, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    start_train = perf_counter()
    
    # Initialize best metrics
    best_train_acc = 0
    best_train_precisions = None
    best_train_recalls = None
    best_train_f1s = None
    best_val_acc = 0
    best_val_precisions = None
    best_val_recalls = None
    best_val_f1s = None
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{max_epochs}', leave=False):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # BCEWithLogitsLoss
            train_loss += loss.item()
            # Compute binary predictions for each class
            preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()  # Shape: (batch_size, num_classes)
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            loss.backward()
            optimizer.step()
        
        train_loss /= len(train_dataloader)
        train_true = np.array(train_true)  # Shape: (num_samples, num_classes)
        train_preds = np.array(train_preds)  # Shape: (num_samples, num_classes)
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_loss /= len(val_dataloader)
        val_true = np.array(val_true)  # Shape: (num_samples, num_classes)
        val_preds = np.array(val_preds)  # Shape: (num_samples, num_classes)
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{max_epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}")
        print(f"Epoch {epoch + 1}/{max_epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}")
        
        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_train_acc = train_acc
            best_train_precisions = train_precisions
            best_train_recalls = train_recalls
            best_train_f1s = train_f1s
            best_val_acc = val_acc
            best_val_precisions = val_precisions
            best_val_recalls = val_recalls
            best_val_f1s = val_f1s
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
            print("Model saved!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    
    total_train_time = perf_counter() - start_train
    return (best_train_acc, best_train_precisions, best_train_recalls, best_train_f1s,
            best_val_acc, best_val_precisions, best_val_recalls, best_val_f1s, total_train_time)

In [9]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].cpu().numpy()  # Shape: (num_classes,)
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = (torch.sigmoid(output) > 0.5).float().cpu().numpy()[0]  # Shape: (num_classes,)
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)  # Shape: (num_samples, num_classes)
    true_labels = np.array(true_labels)  # Shape: (num_samples, num_classes)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [10]:
train_texts = train_df['text'].values
train_labels = train_df.drop(columns=['text']).values

val_texts = val_df['text'].values
val_labels = val_df.drop(columns=['text']).values

test_texts = test_df['text'].values
test_labels = test_df.drop(columns=['text']).values

num_classes = train_labels.shape[1]

# Create datasets
train_dataset = MultiLabelClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiLabelClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiLabelClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

seeds = [2, 3, 5]
batch_sizes = [16, 32]
learning_rates = [5e-5, 3e-5, 2e-5]
results = []

# Grid search loop
for batch_size in batch_sizes:
    for learning_rate in learning_rates:
        for seed in seeds:
            torch.manual_seed(seed)
            model = BertForMultiLabelClassification(num_classes).to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
            criterion = nn.BCEWithLogitsLoss()  # For multi-label classification
            
            train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            val_dataloader = DataLoader(val_dataset, batch_size=batch_size)
            test_dataloader = DataLoader(test_dataset, batch_size=batch_size)
            
            save_path = f'results/bert_multilabel1_bs{batch_size}_lr{learning_rate}_seed{seed}.pt'

            # Train
            if torch.cuda.is_available():
                torch.cuda.reset_max_memory_allocated()

            max_memory_usage_train, retval = memory_usage(
                (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion, save_path),
                 {'max_epochs': MAX_EPOCHS, 'patience': PATIENCE}), max_usage=True, retval=True)
            
            max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

            (train_acc, train_precisions, train_recalls, train_f1s,
             val_acc, val_precisions, val_recalls, val_f1s, total_train_time) = retval
            
            # Load best model
            model.load_state_dict(torch.load(save_path))
            
            # Evaluate
            if torch.cuda.is_available():
                torch.cuda.reset_max_memory_allocated()

            start = perf_counter()
            max_memory_usage_test, test_retval = memory_usage(
                (evaluate_model, (model, test_dataloader), {}), max_usage=True, retval=True)
            total_time_test = perf_counter() - start

            max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

            predictions, true_labels = test_retval
            test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)
            
            # Store individual seed results
            results.append({
                'batch_size': batch_size,
                'learning_rate': learning_rate,
                'seed': seed,
                'train_acc': train_acc,
                'train_precisions': train_precisions.tolist(),
                'train_recalls': train_recalls.tolist(),
                'train_f1s': train_f1s.tolist(),
                'max_memory_usage_train': max_memory_usage_train,
                'max_vram_usage_train': max_vram_usage_train,
                'total_train_time': total_train_time,
                'val_acc': val_acc,
                'val_precisions': val_precisions.tolist(),
                'val_recalls': val_recalls.tolist(),
                'val_f1s': val_f1s.tolist(),
                'test_acc': test_acc,
                'test_precisions': test_precisions.tolist(),
                'test_recalls': test_recalls.tolist(),
                'test_f1s': test_f1s.tolist(),
                'max_memory_usage_test': max_memory_usage_test,
                'max_vram_usage_test': max_vram_usage_test,
                'total_test_time': total_time_test
            })

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
Epoch 1/4:   0%|          | 0/5 [00:00<?, ?it/s]c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.funct

Epoch 1/4 - Train Loss: 0.6160, Acc: 0.2436, F1: [0.18181818 0.2        0.88235294 0.07142857 0.16      ]
Epoch 1/4 - Val Loss: 0.5445, Acc: 0.5000, F1: [0.         0.         0.82352941 0.         0.        ]
Model saved!
Model saved!


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c

Epoch 2/4 - Train Loss: 0.5200, Acc: 0.4615, F1: [0.         0.         0.88571429 0.         0.        ]
Epoch 2/4 - Val Loss: 0.5177, Acc: 0.5000, F1: [0.         0.         0.82352941 0.         0.        ]
Model saved!
Model saved!


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c

...

C:\Users\Rafael\AppData\Local\Temp\ipykernel_21048\1950066938.py:51: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path))
c:\Users\Rafa

Test Time: 0.12 seconds
Test Metrics:
Accuracy: 0.6
F1s: [0.         0.         0.94736842 0.         0.        ]
Precisions: [0.  0.  0.9 0.  0. ]
Recalls: [0. 0. 1. 0. 0.]


Testing: 100%|██████████| 1/1 [00:00<00:00,  8.07it/s]

Test Time: 0.13 seconds
Test Metrics:
Accuracy: 0.6
F1s: [0.         0.         0.94736842 0.         0.        ]
Precisions: [0.  0.  0.9 0.  0. ]
Recalls: [0. 0. 1. 0. 0.]



c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
df = pd.DataFrame(results)
df.to_csv('results/bert_multilabel1.csv', index=False)

In [12]:
df

,batch_size,learning_rate,seed,train_acc,train_precisions,train_recalls,train_f1s,max_memory_usage_train,max_vram_usage_train,total_train_time,...,val_precisions,val_recalls,val_f1s,test_acc,test_precisions,test_recalls,test_f1s,max_memory_usage_test,max_vram_usage_test,total_test_time
0,16,0.00005,2,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1248.843750,2528.906738,5.273854,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1189.027344,1715.291016,1.259908
1,16,0.00005,3,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1268.070312,2539.156738,8.143835,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.113281,1717.791016,1.203630
2,16,0.00005,5,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1268.578125,2542.031738,7.289995,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.488281,1717.916016,1.215830
3,16,0.00003,2,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1268.855469,2549.156738,8.758684,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.671875,1725.541016,1.369954
4,16,0.00003,3,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1268.972656,2547.031738,8.919644,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.789062,1723.416016,1.301652
5,16,0.00003,5,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1269.082031,2548.031738,8.887331,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.816406,1722.916016,1.300475
6,16,0.00002,2,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1269.085938,2546.156738,8.987676,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.820312,1721.541016,1.343361
7,16,0.00002,3,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1269.082031,2546.281738,9.135816,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.816406,1724.416016,1.336011
8,16,0.00002,5,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1269.089844,2542.406738,9.080358,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1272.828125,1715.791016,1.285718
9,32,0.00005,2,0.461538